# 03 — Models, contract, validation, sources

The Pydantic models in `src/coralmap/models/` are the schema; `schema/` is
generated from them. Fields marked with `contract()` are the ones a consumer
can rely on, and `entry.contract_fields()` returns them.

In [ ]:
from pathlib import Path


def repo_root() -> Path:
    """The checkout holding ``entries/``, whether Jupyter started in ``notebooks/`` or the root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "entries").is_dir() and (candidate / "src" / "coralmap").is_dir():
            return candidate
    raise FileNotFoundError("could not find the knowledge-library checkout")


REPO = repo_root()
REPO.name

In [ ]:
from coralmap import load

kl = load(REPO)
kl

## One model class per entry type

`ENTRY_MODELS` maps each entry type to the class the loader validates it
against. Downstream code imports the same classes.

In [ ]:
from coralmap.models import ENTRY_MODELS

for entry_type, cls in ENTRY_MODELS.items():
    print(f"{entry_type.value:22}  {cls.__name__}")

## `contract_fields` and `public_fields`

`contract_fields()` returns only the fields marked `contract()`, including
inside nested models. `public_fields()` returns every field except
`_`-prefixed internal keys. `claims` and `standing` are public but not part of
the contract.

In [ ]:
pathway = kl.pathway("Pathway-5").entry
shipped = pathway.contract_fields(mode="json")
print("contract keys:", sorted(shipped))
print("claims shipped:  ", "claims" in shipped)
print("standing shipped:", "standing" in shipped)

In [ ]:
published = pathway.public_fields(mode="json")
print("claims published:  ", "claims" in published)
print("standing published:", "standing" in published)
print("id:", shipped["id"])

## Edges are derived, not authored

No entry has an `edges` field. A pathway declares `composition_parts` and
`impacts`; a user behavior declares `condition_tags`. `edges(entry)` flattens
those fields into typed edges, which is what `kl.validate()` checks.

In [ ]:
from coralmap.models import edges

for edge in edges(pathway):
    print(f"{edge.kind.value:18}  {edge.source} → {edge.to}")

## Graph checks: `kl.validate()`

`load()` checks each entry against its model. Cross-references can only be
checked once every entry is loaded, so `kl.validate()` is a separate call. It
returns every problem it finds; an empty list means the graph is valid.

In [ ]:
problems = kl.validate()
print(f"{len(problems)} graph problems")
for problem in problems[:8]:
    print(problem)
if len(problems) > 8:
    print(f"… {len(problems) - 8} more")

## Sources

A `claim_evidence` row cites a work by the `kl-` id of a source.
`kl.source(id)` returns that `Source`: its display id, a label carrying the
full title, and its `source_quality` grade. `kl.validate()` reports any
citation whose source is missing.

In [ ]:
from coralmap.vocabularies import EdgeKind

cited = next(
    edge.to for entry in kl.all_entries() for edge in edges(entry) if edge.kind is EdgeKind.citation
)
source = kl.source(cited)
print(source.id, source.display_id)
print(source.label)
print("quality", source.body.source_quality)
print(f"{len(kl.sources)} sources in the library")